<a href="https://colab.research.google.com/github/milleau98/2026-gig-data-analysis/blob/main/google_trends.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pytrends pandas


In [ ]:
from pytrends.request import TrendReq
import pandas as pd
import time

# connect to Google trends
pytrends = TrendReq(hl='en-US', tz=360)

anchor = 'Uber'

keyword_groups = [
    [anchor,'Lyft','DoorDash','Instacart'],
    [anchor, 'Fiverr','Upwork','side hustle','gig'],
    [anchor, 'Herbalife','Primerica','Nu skin', 'USANA'],
    [anchor, 'Etsy', 'Shopify', 'Udemy']
]

all_data = []

for i, group in enumerate(keyword_groups):
  # timeframe and geo
  pytrends.build_payload(group, timeframe='2010-01-01 2025-12-31', geo='US')

  df=pytrends.interest_over_time()

  df=df.drop(columns=['isPartial'])

  # store anchor values
  if i == 0:
    anchor_series = df[anchor]
    all_data.append(df)
  else:
    scale_factor = anchor_series.div(df[anchor]).replace([float("inf"), -float("inf")], 0)
    df = df.multiply(scale_factor, axis=0)
    all_data.append(df.drop(columns=[anchor]))

# combine groups of data
final_data = pd.concat(all_data, axis=1)

final_data.head()

,Uber,Lyft,DoorDash,Instacart,Fiverr,Upwork,side hustle,gig,Herbalife,Primerica,Nu skin,USANA,Etsy,Shopify,Udemy
date,,,,,,,,,,,,,,,
2010-01-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,2.0,1.0,1.0,13.0,0.0,0.0
2010-02-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,2.0,1.0,1.0,13.0,0.0,0.0
2010-03-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,3.0,1.0,1.0,13.0,0.0,0.0
2010-04-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,3.0,0.0,1.0,13.0,0.0,0.0
2010-05-01,2,0,0,0,0.0,0.0,0.0,7.0,3.0,2.0,0.0,1.0,13.0,0.0,0.0


In [ ]:
# convert weekly trends to monthly averages
monthly = final_data.resample("M").mean()

monthly['year'] = monthly.index.year
monthly['month'] = monthly.index.month
monthly['date'] = monthly.index

monthly = monthly.reset_index(drop=True)

monthly.head()

/tmp/ipykernel_28719/1836030185.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly = final_data.resample("M").mean()


,Uber,Lyft,DoorDash,Instacart,Fiverr,Upwork,side hustle,gig,Herbalife,Primerica,Nu skin,USANA,Etsy,Shopify,Udemy,year,month,date
0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,2.0,1.0,1.0,13.0,0.0,0.0,2010,1,2010-01-31
1,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,2.0,1.0,1.0,13.0,0.0,0.0,2010,2,2010-02-28
2,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,3.0,1.0,1.0,13.0,0.0,0.0,2010,3,2010-03-31
3,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,3.0,0.0,1.0,13.0,0.0,0.0,2010,4,2010-04-30
4,2.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,3.0,2.0,0.0,1.0,13.0,0.0,0.0,2010,5,2010-05-31


In [ ]:
# convert weekly trends into quarterly averages
quarterly = final_data.resample("Q").mean()
quarterly.index = quarterly.index.to_period("Q").to_timestamp()
quarterly['year'] = quarterly.index.year
quarterly['quarter'] = quarterly.index.quarter
quarterly['date'] = quarterly.index
quarterly = quarterly.reset_index(drop=True)
quarterly.head()

/tmp/ipykernel_28719/3960241845.py:2: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  quarterly = final_data.resample("Q").mean()


,Uber,Lyft,DoorDash,Instacart,Fiverr,Upwork,side hustle,gig,Herbalife,Primerica,Nu skin,USANA,Etsy,Shopify,Udemy,year,quarter,date
0,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,7.000000,3.000000,2.333333,1.000000,1.000000,13.000000,0.0,0.0,2010,1,2010-01-01
1,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,7.333333,3.000000,2.333333,0.000000,1.000000,13.000000,0.0,0.0,2010,2,2010-04-01
2,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,7.666667,3.000000,2.000000,0.000000,1.000000,15.000000,0.0,0.0,2010,3,2010-07-01
3,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,8.000000,2.666667,2.000000,0.333333,0.666667,18.666667,0.0,0.0,2010,4,2010-10-01
4,2.333333,0.0,0.0,0.0,0.0,0.0,0.0,7.333333,2.666667,2.000000,0.000000,1.000000,24.000000,0.0,0.0,2011,1,2011-01-01


In [ ]:
# convert weekly trends into annual averages
annual = final_data.resample("Y").mean()
annual.index = annual.index.to_period("Y").to_timestamp()
annual["year"] = annual.index.year
annual["date"] = annual.index
annual = annual.reset_index(drop=True)

annual.head()

/tmp/ipykernel_28719/646657277.py:2: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  annual = final_data.resample("Y").mean()


,Uber,Lyft,DoorDash,Instacart,Fiverr,Upwork,side hustle,gig,Herbalife,Primerica,Nu skin,USANA,Etsy,Shopify,Udemy,year,date
0,2.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,7.500000,2.916667,2.166667,0.333333,0.916667,14.916667,0.000000,0.00000,2010,2010-01-01
1,2.083333,0.000000,0.0,0.000000,0.083333,0.0,0.0,7.333333,3.333333,2.083333,0.000000,0.916667,28.416667,0.000000,0.00000,2011,2011-01-01
2,2.666667,0.000000,0.0,0.000000,1.083333,0.0,0.0,6.666667,4.583333,1.916667,0.833333,1.000000,42.750000,1.000000,0.00000,2012,2012-01-01
3,6.500000,1.250000,0.0,0.000000,1.000000,0.0,0.0,6.500000,6.916667,1.916667,1.000000,1.000000,52.158415,1.029897,0.00000,2013,2013-01-01
4,24.500000,4.416667,0.0,0.666667,1.083333,0.0,0.0,6.333333,7.000000,1.833333,0.666667,0.833333,55.661437,1.905634,1.03615,2014,2014-01-01


In [ ]:
# add function to create lagged columns for given dataframe

#def add_lag(df, lag=1):
#  """
#  Adds lagged columns to a df for all columns except 'year', 'month', 'date'

 # Returns:
  #dataframe with lagged columns appended
  #"""

  #lagged_df = df.copy()

  #cols_to_lag = [c for c in df.columns if c not in ['year', 'month', 'date']]

  #for col in cols_to_lag:
   # lagged_df[f'{col}_lag'] = df[col].shift(lag)

 # return lagged_df

#monthly_lagged = add_lag(monthly, lag=1)
#quarterly_lagged = add_lag(quarterly, lag=1)
#annual_lagged = add_lag(annual, lag=1)

In [ ]:
import os
os.makedirs("data/google_trends", exist_ok=True)

# create dataset files
monthly.to_csv('data/google_trends/google_trends_monthly.csv', index=False)
quarterly.to_csv('data/google_trends/google_trends_quarterly.csv', index=False)
annual.to_csv("data/google_trends/google_trends_annual.csv", index=False)

In [ ]:
# download as csv
from google.colab import files
files.download('data/google_trends/google_trends_monthly.csv')
files.download('data/google_trends/google_trends_quarterly.csv')
files.download('data/google_trends/google_trends_annual.csv')




<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>